Предобработка данных о погоде (weather)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

df = pd.read_csv('weather.csv', na_values=['', '#DIV/0!'])
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print("Исходный размер датасета:", df.shape)
display(df.head(10))

Удаление строк с пропущенной датой

In [ ]:
len_before = len(df)
df = df.dropna(subset=['datetime'])
print(f"Строк без datetime: удалено {len_before - len(df)} ({100*(len_before - len(df))/len_before:.2f}%)")

df['datetime'] = pd.to_datetime(df['datetime'])

Преобразование числовых колонок

In [ ]:
numeric_cols = ['temperature', 'precipitation_total', 'wind_gust',
                'wind_speed', 'cloud_cover_total', 'sunshine_duration']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

Анализ выбросов в погодных переменных

In [ ]:
for col in numeric_cols:
    print(f"\n=== {col} ===")
    print(df[col].describe(percentiles=[.01, .05, .95, .99]))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, col in enumerate(numeric_cols):
    row, col_idx = divmod(i, 3)
    sns.boxplot(x=df[col], ax=axes[row, col_idx])
    axes[row, col_idx].set_title(col)
plt.tight_layout()
plt.show()

len_before = len(df)
df = df[(df['temperature'] >= -30) & (df['temperature'] <= 50)]
df = df[df['precipitation_total'] >= 0]
df = df[df['wind_speed'] <= 150]
df = df[(df['cloud_cover_total'] >= 0) & (df['cloud_cover_total'] <= 100)]

Интерполяция пропусков и удаление дубликатов

In [ ]:
df = df.set_index('datetime').interpolate(method='time').reset_index()

len_before = len(df)
df = df.drop_duplicates(subset=['datetime'])

print("\nИтоговый размер датасета после очистки:", df.shape)
display(df.head(10))

Сохранение очищенных данных

In [ ]:
df.to_csv('weather_clean.csv', index=False)